In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("IMDB Dataset.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

# Text Pre-Processing

### 1. Converting to lowercase

In [ ]:
df["review"] = df["review"].str.lower()

### 2. Removing the URL's

In [ ]:
import re   #regular expression lib

def remove_urls(text):  # created a fnx for removing chars
    text = re.sub( r"https\S+", "", text)  # (pattern, replace, String) is the flow where \S is for non-white space char(no spaces only char) and \s is for white space char(no char only spaces )
    return text
    
df["review"] = df["review"].apply(remove_urls)

### 3. Removing punctuations

In [ ]:
def remove_punc(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text

df["review"] = df["review"].apply(remove_punc)
df.head()

### 4. Removing HTML tags

In [ ]:
def remove_html(text):
    text = re.sub(r"<.*?>", "", text)
    return text

df["review"] = df["review"].apply(remove_html)

### 5. Removing stopwords

In [ ]:
import nltk 

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def remove_stopwords(text):
    tokens = word_tokenize(text)  # eg, i/p = "my name is syed maaz", o/p = "my", "name", "is", "syed", "maaz"
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")
    return text

df["review"] = df["review"].apply(remove_stopwords)
df.head()

### 6. Stemming

In [ ]:
from nltk.stem import PorterStemmer

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text) #tokennise our text 
    for word in tokens:
        stemmed_word = ps.stem(word)
        stemmed_words.append(stemmed_word)
        
    return " ".join(stemmed_words)  # to convert list to str
    
df["review"] = df["review"].apply(stemming)
df.head()

### 7. Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

y = df["sentiment"]

y.head()

### 8. Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf_idf = TfidfVectorizer(max_features=5000)  # max_features bcoz our system will crash for big values

X = tf_idf.fit_transform(df["review"])

# Dataset and Dataloader

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
# converted it back to arr
X_train = X_train.toarray()
X_test = X_test.toarray()

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

In [ ]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=True)

# RNN

In [ ]:
import torch.nn as nn
import torch.optim as optim

In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_layers=1, neurons=128):
        super().__init__()

        self.hidden_layers = hidden_layers
        self.neurons = neurons

        # RNN layer
        self.rnn = nn.RNN(input_size, neurons, hidden_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(neurons, 1) # i/p features, o/p features (converts 128 neurons to 1)

    def forward(self, x):
        # optional => init hidden_state with 0 check notes (num of hidden layers, batch size, neurons)
        h0 = torch.zeros(self.hidden_layers, x.size(0), self.neurons)
        
        # 1st value = hidden state of all the timesteps => (batch, seq_len, neurons)
        # 2nd value = final hidden state of last timestep (_ we ignore this)
        output, _ = self.rnn(x, h0)

        output = self.fc(output[:, -1, :]) # take all batches, last time step bcoz it contains the info of whole sequence, all neurons 
        return output

    

In [ ]:
input_size = X_train.shape[1]  # GPT this line

model = RNN(input_size)

criterion = nn.BCELoss() # because it is a bin classification
optimizer = optim.Adam(model.parameters())

# Training the RNN

In [ ]:
# squeeze => remove dimension
# unsquueze => add dimension

epochs = 10

for epoch in range(epochs):
    model.train()
    epoch_training_loss = 0.0

    for xb, yb in train_dataloader:
        optimizer.zero_grad()

        xb = xb.unsqueeze(1) # out model needs 3d but we have 2d so we added 1 dim to make it 3d
        outputs = model(xb) # this is 2d (batch_sise, 1) but for sigmoid we need 1d
        outputs = torch.sigmoid(outputs.squeeze()) # this is now 1d (batch_size, )

        loss = criterion(outputs, yb) # compute loss
        loss.backward()
        optimizer.step() # update weights

        epoch_training_loss += loss.item()
    print(f"Epoch: {epoch} / {epochs} & Loss: {epoch_training_loss / len(train_dataloader)}")

In [ ]:
# Evaluate

model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_dataloader:
        xb = xb.unsqueeze(1)
        outputs = model(xb)

        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float() # if predicted is greater than 0.5 convert it to 1

        correct += (predicted == yb).sum().item()
        total += yb.size(0)
    print(f"Accuracy: {correct / total}")